In [11]:
import sys
sys.path.append('desc_utils')

from desc_utils import  atom_numbering as an
import pandas as pd
import orca_descriptors as od
import os
import numpy as np
import tqdm as tqdm
from datetime import datetime

In [12]:
#Provide path to directory or directory of directories containing xyz files
# path can be absolute, e.g. "/home/aconner/orca_descriptors/library_pyox_2026_03_03/",
# or relative, e.g. "library_pyox_2026_03_03" (joined to current working directory).
src_dir = "DATA/library_pyox_2026_03_03"  # change this as needed
if os.path.isabs(src_dir):
    path = os.path.abspath(src_dir)
else:
    path = os.path.abspath(os.path.join(os.getcwd(), src_dir))
# ensure trailing slash for compatibility with existing code
if not path.endswith(os.sep):
    path = path + os.sep


In [13]:
#Define lists for decriptor retrieval 
atoms = ["H1", "H2", "C1", "C2", "N1", "N2", "Ni"]
bonds = [("C1", "N1"), ("N1", "C1"), ("C2", "N2"), ("N2", "C2"), ("Ni", "N1"), ("Ni", "N2"), ("Ni", "H1"), ("Ni", "H2")] #if order matters for 2-atom descriptors (e.g. Sterimol B1), atoms must be listed in desired order
angles = [("H2", "Ni", "H1")]
dihedrals = [("N1", "C1", "C2", "N2"), ("C2", "N2", "Ni", "H2"), ("C1", "N1", "Ni", "H1")]

#quadrants = [[2, 1, 3, 4, 6]] 


In [14]:
#generate descriptor dictionary for each row in dataframe and add to dataframe
def get_descriptors(df):
    results_dict = {}
    for row in df.itertuples():

        P = od.PropertiesXYZ(path=df.loc[row.Index, "path"])
        
        #Zero atom properties
        P.get_SASA_and_Volume()

        #One atom properties
        atom_list = []
        for atom in atoms:
            atom_list.append(int(df.loc[row.Index, atom]))
        
        P.define_atom_list(atom_list)
        P.get_Vbur()
        P.get_atomic_SASA()

        #Two atom properties
        bond_list = []
        for bond in bonds: 
            bond_list.append([int(df.loc[row.Index, bond[0]]), int(df.loc[row.Index, bond[1]])])
        P.define_bond_list(bond_list)
        P.get_bond_distances()
        P.get_Sterimol_parameters(morfeus=True)

        #Three atom properties
        angle_list = []
        for angle in angles:
            angle_list.append([int(df.loc[row.Index, angle[0]]), int(df.loc[row.Index, angle[1]]), int(df.loc[row.Index, angle[2]])])
        P.define_angle_list(angle_list)
        P.get_angles()

        #Four atom properties
        dihedral_list = []
        for dihedral in dihedrals:
            dihedral_list.append([int(df.loc[row.Index, dihedral[0]]), int(df.loc[row.Index, dihedral[1]]), int(df.loc[row.Index, dihedral[2]]), int(df.loc[row.Index, dihedral[3]])])
        P.define_dihedral_list(dihedral_list)
        P.get_dihedrals()
        #P.get_pyramidalizations()

        #Quadrant properties
        #quadrant_list = quadrants
        #P.define_quadrant_list(quadrant_list)
        #P.get_vbur_quadrants()
        #P.get_vbur_octants()

        results_dict[row.Index] = P.results
    return results_dict



In [15]:
def make_dict_atoms(df):
    dict_atoms = {}
    for row in df.itertuples():
        dict_atoms[row.Index] = {}
        for col in df.columns:
            if col.startswith("H1"):
                atom = col[:-1] 
                number = df.at[row.Index, col]
                dict_atoms[row.Index][col] = atom + str(number)
            elif col.startswith("H2"):
                atom = col[:-1] 
                number = df.at[row.Index, col]
                dict_atoms[row.Index][col] = atom + str(number)
            elif col.startswith("C1"):
                atom = col[:-1] 
                number = df.at[row.Index, col]
                dict_atoms[row.Index][col] = atom + str(number)
            elif col.startswith("C2"):
                atom = col[:-1] 
                number = df.at[row.Index, col]
                dict_atoms[row.Index][col] = atom + str(number)
            elif col.startswith("N1"):
                atom = col[:-1] 
                number = df.at[row.Index, col]
                dict_atoms[row.Index][col] = atom + str(number)
            elif col.startswith("N2"):
                atom = col[:-1] 
                number = df.at[row.Index, col]
                dict_atoms[row.Index][col] = atom + str(number)
            elif col.startswith("Ni"):
                atom = col  
                number = df.at[row.Index, col]
                dict_atoms[row.Index][col] = atom + str(number)
    return dict_atoms


In [16]:
#alter key in results dictionary to match format of user defined lists and add to dataframe
#Have to add new descriptors to this code!!
def alter_results_dict(results_dict, dict_atoms):
    new_results = {}
    for key in results_dict.keys():
        new_results[key] = {}
        for subkey in results_dict[key].keys():
            if subkey.startswith("Bond_"):
                for bond in bonds:
                    bond_key = [dict_atoms[key][bond[0]], dict_atoms[key][bond[1]]]
                    if all(bk in subkey for bk in bond_key):
                        new_key = "Bond_" + bond[0] + "_" + bond[1]
                        new_results[key][new_key] = results_dict[key][subkey]
            elif subkey.startswith("Sterimol_") and subkey.endswith("_L"):
                for bond in bonds:
                    bond_key = [dict_atoms[key][bond[0]], dict_atoms[key][bond[1]]]
                    if all(bk in subkey for bk in bond_key):
                        new_key = "Sterimol_" + bond[0] + "_" + bond[1] +"_L"
                        new_results[key][new_key] = results_dict[key][subkey]
            elif subkey.startswith("Sterimol_") and subkey.endswith("_Bmin"):
                for bond in bonds:
                    bond_key = [dict_atoms[key][bond[0]], dict_atoms[key][bond[1]]]
                    if all(bk in subkey for bk in bond_key):
                        new_key = "Sterimol_" + bond[0] + "_" + bond[1] +"_Bmin"
                        new_results[key][new_key] = results_dict[key][subkey]
            elif subkey.startswith("Sterimol_") and subkey.endswith("_Bmax"):
                for bond in bonds:
                    bond_key = [dict_atoms[key][bond[0]], dict_atoms[key][bond[1]]]
                    if all(bk in subkey for bk in bond_key):
                        new_key = "Sterimol_" + bond[0] + "_" + bond[1] +"_Bmax"
                        new_results[key][new_key] = results_dict[key][subkey]
            elif subkey.startswith("Angle_"):
                for angle in angles:
                    angle_key = [dict_atoms[key][angle[0]], dict_atoms[key][angle[1]], dict_atoms[key][angle[2]]]
                    if all(ak in subkey for ak in angle_key):
                        new_key = "Angle_" + angle[0] + "_" + angle[1] + "_" + angle[2]
                        new_results[key][new_key] = results_dict[key][subkey]
            elif subkey.startswith("Dihed_"):
                for dihedral in dihedrals:
                    dihedral_key = [dict_atoms[key][dihedral[0]], dict_atoms[key][dihedral[1]], dict_atoms[key][dihedral[2]], dict_atoms[key][dihedral[3]]]
                    if all(dk in subkey for dk in dihedral_key):
                        new_key = "Dihed_" + dihedral[0] + "_" + dihedral[1] + "_" + dihedral[2] + "_" + dihedral[3]
                        new_results[key][new_key] = results_dict[key][subkey]
            elif subkey.startswith("%Vbur_atom_"):
                for atom in atoms:
                    atom_key = dict_atoms[key][atom]
                    if atom_key in subkey:
                        new_key = "%Vbur_" + atom
                        new_results[key][new_key] = results_dict[key][subkey]
            elif subkey.startswith("SASA_"):
                for atom in atoms:
                    atom_key = dict_atoms[key][atom]
                    if atom_key in subkey:
                        new_key = "SASA_" + atom
                        new_results[key][new_key] = results_dict[key][subkey]
            else:
                new_results[key][subkey] = results_dict[key][subkey]
    return new_results



In [17]:
#add new results to dataframe
def results_to_df(df, new_results):
    for key in new_results.keys():
        for subkey in new_results[key].keys():
            df.at[key, subkey] = new_results[key][subkey]
    return df

In [18]:
def descriptor_stats(df):
    #Find min, max, and average of descriptors across dataset
    descriptor_stats = {}

    #boltzmann weighted average
    #convert energy to relative energy
    if 'Relative_energy' not in df.columns:
        df.insert(5, 'Relative_energy', df['SCF_energy'] - df['SCF_energy'].min())
    else:
        df['Relative_energy'] = df['SCF_energy'] - df['SCF_energy'].min()


    T = 298.15 #temperature in K
    R = 3.1668105e-6 #Boltzmann constant in eH/K
    weights = []

    for row in df.itertuples():
        energy = row.Relative_energy
        weight = np.exp(-energy/(R*T))
        weights.append(weight)

    normalized_weights = weights / np.sum(weights)

    for column in df.iloc[:, 15:]:
        descriptor_stats[f'{column}_min'] = df[column].min()
        descriptor_stats[f'{column}_max'] = df[column].max()
        descriptor_stats[f'{column}_mean'] = np.average(df[column], weights=normalized_weights)

    descriptor_stats_df = pd.DataFrame(descriptor_stats, index=[0])
    if df["smiles"].nunique() == 1:
        descriptor_stats_df.insert(0, "SMILES", df["smiles"].iloc[0])
        descriptor_stats_df.insert(1, "INCHI", df["inchi"].iloc[0])
    else:
        print("Multiple SMILES strings found in dataset. Please seperate before calculating descriptor statistics.")    

    return descriptor_stats_df


In [19]:
total_df = pd.DataFrame()

with os.scandir(path) as entries:
    for entry in entries:
        if entry.is_dir():
            sub_path = os.path.join(path, entry.name)
            if not sub_path[-1] == os.sep:
                sub_path = sub_path + os.sep
            df = an.compile_final(sub_path)
            dict_atoms = make_dict_atoms(df)
            results_dict = get_descriptors(df)
            new_results = alter_results_dict(results_dict, dict_atoms)
            df = results_to_df(df, new_results)
            descriptor_stats_df = descriptor_stats(df)
            total_df = pd.concat([total_df, descriptor_stats_df], ignore_index=True)

# derive optional DATA subfolder name
folder_name = ""
if "DATA" in path.split(os.sep):
    parts = path.split(os.sep)
    idx = parts.index("DATA")
    if idx + 1 < len(parts) and parts[idx + 1]:
        folder_name = parts[idx + 1]

date_str = datetime.now().strftime("%Y_%m_%d")
filename_stem = f"{date_str}_{folder_name + '_' if folder_name else ''}descriptor_stats.csv"

print(f"Writing output to {filename_stem}")
total_df.to_csv(filename_stem, index=False)

filename_stem

No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified 

'2026_03_20_library_pyox_2026_03_03_descriptor_stats.csv'

In [20]:
total_df.to_csv("descriptor_stats.csv", index=False)